In [1]:
import pandas as pd
import re
from datetime import datetime, date, timedelta
from calendar import monthrange

In [2]:
df= pd.DataFrame()
df= pd.read_json("../data/processed/processed_events_2026-02-08.json")
df.shape

(9991, 14)

In [3]:
mask = df['first_date']> datetime.now().isoformat()
df[mask].shape[0]

717

In [4]:
def normalize_str(text:str) -> str:
    location = text.strip().lower()
    """ Remove accents from text """
    accents = { 'a': ['à', 'ã', 'á', 'â'],
                'e': ['é', 'è', 'ê', 'ë'],
                'i': ['î', 'ï'],
                'u': ['ù', 'ü', 'û'],
                'o': ['ô', 'ö'],
                ' ': ['-','/'] 
                }
    for (char, accented_chars) in accents.items():
        for accented_char in accented_chars:
            location = location.replace(accented_char, char)
    return location    

In [5]:

def parse_date(query: str):
    """Extract date from query"""
    months = ['janvier',"fevrier", "mars", "avril", "mai","juin", "juillet","aout", "septembre", "octobre", "novembre", "decembre"]
    query_lower = normalize_str(query)
    today = today = datetime.today()
    print(query_lower)
    #On a precise date
    match = re.search(r'\ble\s+(0?[1-9]|[12][0-9]|3[01])[\/\- ](0?[1-9]|1[0,1,2])[\/\-\s ](?:20|)([234][0-9]|)', query_lower +" ")
    if match and match.lastindex >= 2: # pyright: ignore[reportOptionalOperand]
        dd = int(match[1])
        mm = int(match[2])
        aa = int("20" + match[3] if match[3] else datetime.now().strftime('%y'))
        return (datetime(aa,mm,dd),0)

    # Today/tonight
    if re.search(r'\b(ce soir|aujourd\'hui|ce jour|ce matin|cet[te]* apres[\- ]midi)\b', query_lower):
        return (today,0)
    
    # Tomorrow
    if re.search(r'\b(demain)\b', query_lower):
        return (today+ timedelta(days=1), 0)
    
    # Relative days: "dans X jours"
    match = re.search(r'\bdans\s+(\d+)\s+(jour|jours)\b', query_lower)
    if match:
        days = int(match.group(1))
        return (today + timedelta(days=days), 0)
    
    # This weekend
    if re.search(r'\b(ce week[\- ]?end)\b', query_lower):
        days_until_saturday = (5 - today.weekday()) % 7 or 7
        
        return (today + timedelta(days=days_until_saturday) ,1)
    
    # Next week
    if re.search(r'\b(semaine\sprochaine)\b', query_lower):
        days_until_monday = (- today.weekday()) % 7 or 7
            
        return (today + timedelta(days=days_until_monday) ,6)
    
    # Current week
    if re.search(r'\b(?:cette|la)\s(semaine)\b', query_lower):
        days_until_sunday = (6 - today.weekday()) % 7 
            
        return (today  , days_until_sunday)
    
    # On precise month
    current_year = int(today.strftime("%Y"))
    for index, month in enumerate(months):
        if re.search(fr"\b(?:mois de\s|mois d'|en\s)({month})\b", query_lower):
            index_month = (index + 1) 
            first_day = datetime(current_year, index_month,1)
            first_day = max( first_day,today)
            days_in_months = monthrange(first_day.year, first_day.month)[1] - first_day.day
            return (first_day ,days_in_months)
            

    # dafault : return the next 30 days
    return (today, 30)


In [6]:

query_lower = "je voudrais danser le 14-2-26 "
print(parse_date(query_lower))


je voudrais danser le 14 2 26
(datetime.datetime(2026, 2, 14, 0, 0), 0)


In [7]:
query_lower = normalize_str("je voudrais danser le 14-2-26 ")
print(query_lower)
match = re.search(r'\ble\s+(0?[1-9]|[12][0-9]|3[01])[\/\- ](0?[1-9]|1[0,1,2])[\/\-\s ](?:20|)([234][0-9]|)', query_lower +" ")
if match and match.lastindex >= 2: 
    dd = int(match[1])
    mm = int(match[2])
    aa = int("20" + match[3] if match[3] else datetime.now().strftime('%y'))
    dateok = (datetime(aa,mm,dd),0)
if match:
    print(match,match.lastindex, dateok)

je voudrais danser le 14 2 26
<re.Match object; span=(19, 29), match='le 14 2 26'> 3 (datetime.datetime(2026, 2, 14, 0, 0), 0)


In [8]:
dt = date(2026,2,8)
pattern = 'emploi'
selected_ids = []
for index,data in df.iterrows():
    has_pattern = pattern in data['title_fr'].lower() or pattern in data['description_fr'].lower() or pattern in data['longdescription_fr'].lower() 
    if has_pattern:
        for timing in data['timings']:
            date_timing = datetime.fromisoformat(timing['begin']).date()
            if date_timing > dt:
                selected_ids.append(data['uid'])
                break
if selected_ids :
    print(f"Réponse(s) trouvées : {len(selected_ids)}")
else:
    print("Pas de réponses")

Réponse(s) trouvées : 40


In [ ]:
selected_events = df[df['uid'].isin(selected_ids)]
if not selected_events.empty:
    for idx, data in selected_events.iterrows():
        display(f"uid : {data['uid']}")
        display(f"title_fr : {data['title_fr']}")
        display(f"description_fr : {data['description_fr']}")
        display(f"longdescription_fr : {data['longdescription_fr']}")
        display(f"timings : {data['timings']}")                        

In [10]:
mask = df["first_date"] >= '2026-02-09T00:00:00+01:00'
df_2026 = df[mask].sort_values('first_date')
df_2026

,uid,title_fr,description_fr,longdescription_fr,timings,first_date,location_name,location_city,location_department,location_address,canonicalurl,conditions_fr,location_lat,location_lon
3099,37506723,Semaine du Tourisme 2026 : Rencontrez un recru...,Le secteur de l'Hôtellerie Café restauration (...,<p>Le secteur de l'Hôtellerie Café restauratio...,"[{'begin': '2026-02-09T08:45:00+01:00', 'end':...",2026-02-09T08:45:00+01:00,Uzès - Agence NIMES COURBESSAC,Uzès,Gard,30700 Uzès,https://openagenda.com/semaine-des-metiers-du-...,,44.002945,4.416701
2768,34606071,EMPLOIS SAISONNIERS A SAINT-TROPEZ,Emplois saisonniers à Saint -Tropez. Rencontre...,"<p>Pour la 4ème année consécutive, des jeunes ...","[{'begin': '2026-02-09T09:00:00+01:00', 'end':...",2026-02-09T09:00:00+01:00,Mission Locale Tarn Sud,Castres,Tarn,46 avenue Albert 1er 81100 Castres,https://openagenda.com/semaine-des-metiers-du-...,,43.601855,2.237384
2604,33330858,Devenez Conducteur de bus à la TaM : Saisissez...,Participez à notre journée de sélection pour i...,<p>Participez à notre journée de sélection pou...,"[{'begin': '2026-02-09T09:00:00+01:00', 'end':...",2026-02-09T09:00:00+01:00,Pérols - Agence MONTPELLIER MEDITERRANEE,Pérols,Hérault,34470 Pérols,https://openagenda.com/semaine-des-metiers-du-...,,43.582291,3.938849
9147,92360051,VENEZ RENCONTRER LES EMPLOYEURS DE L'HOTELLERI...,Rencontre avec des employeurs de l'hotellerie ...,<p>Rencontre avec des employeurs de l'hoteller...,"[{'begin': '2026-02-09T09:00:00+01:00', 'end':...",2026-02-09T09:00:00+01:00,Agence FIGEAC,Figeac,Lot,46100 Figeac,https://openagenda.com/semaine-des-metiers-du-...,,44.608360,2.024219
9464,95111102,Décrochez votre EMPLOI SAISONNIER en LOZERE Hô...,Un Job Dating spécialement dédié aux opportuni...,<p>Un Job Dating spécialement dédié aux opport...,"[{'begin': '2026-02-09T09:15:00+01:00', 'end':...",2026-02-09T09:15:00+01:00,La Malène - Agence MENDE,La Malène,Lozère,48210 La Malène,https://openagenda.com/francetravail/events/de...,,44.302639,3.319532
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1307,21828241,Domaine de Lascroux,R avec hebergement,<p>Le Domaine de Lascroux est le centre de vac...,"[{'begin': '2032-01-01T02:00:00+01:00', 'end':...",2032-01-01T02:00:00+01:00,Domaine de Lascroux,Puycelsi,Tarn,chemin de las croux 81140 Puycelsi,https://openagenda.com/catalogue-departemental...,,43.994839,1.714027
1302,21791189,Domaine du Ventouzet,Centre d'accueil de groupes,"<p>C’est au pied de l’Aubrac, en Lozère à 1100...","[{'begin': '2032-01-01T02:00:00+01:00', 'end':...",2032-01-01T02:00:00+01:00,Domaine du Ventouzet,Peyre en Aubrac,Lozère,le ventouzet 48100 peyre en Aubrac,https://openagenda.com/catalogue-structures-ac...,,44.665408,3.234638
7207,74474219,Centre Lozère Evasion - centre Vallée de la Tr...,Centre d'Hébergement,"<p>COLONIES DE VACANCES, CLASSES VERTES ET SÉJ...","[{'begin': '2032-01-01T02:00:00+01:00', 'end':...",2032-01-01T02:00:00+01:00,Centre Lozère Evasion,Le Malzieu-Ville,Lozère,Route de St Alban 48140 Le Malzieu Ville,https://openagenda.com/catalogue-departemental...,,44.851083,3.346617
7078,73490856,EARL FERME EQUESTRE PUECH MERLHOU,R,"<h2>DESCRIPTION DU CENTRE D’HEBERGEMENT, DES I...","[{'begin': '2032-01-01T02:00:00+01:00', 'end':...",2032-01-01T02:00:00+01:00,FERME EQUESTRE PUECH MERLHOU,Saint-Grégoire,Tarn,81350 SAINT GREGOIRE,https://openagenda.com/catalogue-structures-ac...,,43.961700,2.259720


In [11]:
df.columns

Index(['uid', 'title_fr', 'description_fr', 'longdescription_fr', 'timings',
       'first_date', 'location_name', 'location_city', 'location_department',
       'location_address', 'canonicalurl', 'conditions_fr', 'location_lat',
       'location_lon'],
      dtype='object')

In [12]:
today = datetime.today()
today.strftime("%A %-d %B %Y")

'Wednesday 11 February 2026'

In [13]:
ligne = "Hérault"
accents = { 'a': ['à', 'ã', 'á', 'â'],
                    'e': ['é', 'è', 'ê', 'ë'],
                    'i': ['î', 'ï'],
                    'u': ['ù', 'ü', 'û'],
                    'o': ['ô', 'ö'] }
for (char, accented_chars) in accents.items():
    for accented_char in accented_chars:
        ligne = ligne.replace(accented_char, char)

In [14]:
ligne

'Herault'